In [ ]:
!nvidia-smi

Mon Jul 21 22:47:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:E3:00.0 Off |                    0 |
| N/A   35C    P8             36W /  350W |       1MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from datasets import load_dataset, DatasetDict

from transformers import AutoTokenizer
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

In [ ]:
data = load_dataset(
            "allenai/c4", "en", split="train", streaming=True
        )
val_data = load_dataset(
            "allenai/c4", "en", split="validation", streaming=True
        ) 

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

In [6]:
import torch
from torch.utils.data import IterableDataset, get_worker_info

class PreprocessedIterableDataset(IterableDataset):
    def __init__(self, data, tokenizer, batch_size, max_length):
        super().__init__()
        self.data = data
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.max_length = max_length

    def __iter__(self):
        iter_data = iter(self.data)

        batch = []
        for example in iter_data:
            tokenized_example = self.tokenizer(
                example["text"],
                max_length=self.max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            batch.append(tokenized_example)

            if len(batch) == self.batch_size:
                yield self._format_batch(batch)
                batch = []

        if batch:
            yield self._format_batch(batch)

    def _format_batch(self, batch):
        input_ids = torch.stack([item["input_ids"].squeeze(0) for item in batch])
        attention_mask = torch.stack([item["attention_mask"].squeeze(0) for item in batch])

        return {"input_ids": input_ids, "attention_mask": attention_mask}

In [ ]:
max_length = 256
device = f"cuda:0"


tokenizer = AutoTokenizer.from_pretrained(
    "t5-base", model_max_length=max_length
)
 

# only used in eval ...
def preprocess_batched(batch):
    batch = tokenizer(
        batch["text"],
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    return batch

In [27]:
batch_size = 128
workers = 4

dataset = PreprocessedIterableDataset(
                data, tokenizer, batch_size=batch_size, max_length=max_length
            )


dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=None, num_workers=workers,
    #pin_memory=True,          # Enables pinned memory
    #persistent_workers=True,   # Keeps workers alive across epochs (optional but helps)
) 

In [28]:
from modeling_llama import LlamaForCausalLM

model_config = 'llama_60m.json'
model_config = AutoConfig.from_pretrained(model_config)

model = LlamaForCausalLM(model_config).to(device)

n_total_params = sum(p.numel() for p in model.parameters())
trainable_params = [p for p in model.parameters() if p.requires_grad]

print(n_total_params)


58073600


In [29]:
import transformers

lr = 1e-3
weight_decay = 0.0 
warmup_steps = 1000


optimizer = torch.optim.Adam(
            trainable_params, lr=lr, weight_decay=weight_decay
        )

scheduler = transformers.get_constant_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            last_epoch=-1,            
        )

In [30]:
from loguru import logger

global_step = 0
update_step = 0
tokens_seen = 0
tokens_seen_before = 0
world_size = 1

num_training_steps = 10000
pad_idx = tokenizer.pad_token_id

total_batch_size = 256
gradient_accumulation = None
grad_clipping = 0.0 

if  total_batch_size is not None:
    if  gradient_accumulation is None:
        assert (
            total_batch_size % world_size == 0
        ), "total_batch_size must be divisible by world_size"
        gradient_accumulation = total_batch_size // (
            batch_size * world_size
        )
        # logger.info(f"{args.gradient_accumulation}-{world_size}-{args.total_batch_size}-{args.batch_size}")
        assert (
            gradient_accumulation > 0
        ), "gradient_accumulation must be greater than 0"

assert (
    gradient_accumulation * batch_size * world_size
    == total_batch_size
), "gradient_accumulation * batch_size * world_size must be equal to total_batch_size"



for batch_idx, batch in enumerate(dataloader):

    print(batch_idx)
 
    global_step += 1
    

    if update_step > num_training_steps:
        logger.info(
            f"Reached max number of update steps (f{num_training_steps}). Stopping training."
        )
        break
    

    batch = {k: v.to(device) for k, v in batch.items()}
    labels = batch["input_ids"].clone()
    labels[labels == pad_idx] = -100
    tokens_seen += (batch["input_ids"] != pad_idx).sum().item() * world_size

 
    loss = model(**batch, labels=labels).loss
    scaled_loss = loss / gradient_accumulation
    scaled_loss.backward()    


    if global_step % gradient_accumulation != 0:
        continue

    #######
    if grad_clipping != 0.0:
        torch.nn.utils.clip_grad_norm_(trainable_params, grad_clipping)
        
    #pbar.update(1)


    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
        
    update_step += 1
 
 

# ##############################
# END of training loop
# ##############################
logger.info("Training finished") 


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


KeyboardInterrupt: 